## Setup e importaciones

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('./data/product_activity.csv')

print("Dataset cargado")
print(f"Shape: {df.shape}")

Dataset cargado
Shape: (8782, 12)


## Primera fotografia

In [ ]:
#ver primeras 3 filas
print("------Head------")
display(df.head(3))

#mostrar informacion general del dataset
print("\n------Info------")
df.info()

print("\n------Describe------")
display(df.describe())

------Head------


,user_id,created_at,country,plan_type,user_age,post_id,post_category,post_created_at,votes_received,user_total_posts,days_since_signup,device_type
0,U01988,2025-02-18T02:07:44,PY,pro,26.0,P0008515,sport,2025-05-07T20:55:28,7,16,78,mobile
1,U00236,2025-06-22T07:49:10,BR,free,27.0,P0001023,tech,2025-09-13T20:31:06,1,9,83,web
2,U00791,2024-02-12T02:45:45,CL,free,28.0,P0003405,tech,2024-02-14T05:17:48,11,2,2,mobile



------Info------
<class 'pandas.DataFrame'>
RangeIndex: 8782 entries, 0 to 8781
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            8782 non-null   str    
 1   created_at         8782 non-null   str    
 2   country            8782 non-null   str    
 3   plan_type          8782 non-null   str    
 4   user_age           8028 non-null   float64
 5   post_id            8782 non-null   str    
 6   post_category      8782 non-null   str    
 7   post_created_at    8782 non-null   str    
 8   votes_received     8782 non-null   int64  
 9   user_total_posts   8782 non-null   int64  
 10  days_since_signup  8782 non-null   int64  
 11  device_type        8782 non-null   str    
dtypes: float64(1), int64(3), str(8)
memory usage: 823.4 KB

------Describe------


,user_age,votes_received,user_total_posts,days_since_signup
count,8028.000000,8782.000000,8782.000000,8782.000000
mean,27.902591,6.918356,8.324186,29.479390
std,7.547052,5.127311,6.754906,36.819928
min,16.000000,0.000000,1.000000,0.000000
25%,22.000000,3.000000,4.000000,5.000000
50%,28.000000,6.000000,6.000000,17.000000
75%,33.000000,9.000000,11.000000,40.000000
max,58.000000,74.000000,39.000000,404.000000


## Nulos, duplicados y valores unicos

In [12]:
#calcular nulos
print("--- Nulos por Columna ---")
nulos = df.isnull().sum()
#porcentaje de nulos
nulos_pct = (nulos / len(df) * 100).round(2)
print(pd.DataFrame({
    'nulos': nulos,
    '%': nulos_pct
}))


#calcular los duplicados exactos
print("\n--- Duplicados Exactos ---")
print(f"Filas Duplicadas: {df.duplicated().sum()}")


#valores unicos en columnas categoricas
print("\n--- Valores Unicos ---")
for col in ['plan_type','post_category','device_type']:
    print(f"\n {col}:")
    print(df[col].value_counts())


--- Nulos por Columna ---
                   nulos     %
user_id                0  0.00
created_at             0  0.00
country                0  0.00
plan_type              0  0.00
user_age             754  8.59
post_id                0  0.00
post_category          0  0.00
post_created_at        0  0.00
votes_received         0  0.00
user_total_posts       0  0.00
days_since_signup      0  0.00
device_type            0  0.00

--- Duplicados Exactos ---
Filas Duplicadas: 172

--- Valores Unicos ---

 plan_type:
plan_type
free            5978
pro             1460
enterprise       306
Free             208
 free            197
FREE             196
FreE             189
PRo               46
 pro              44
PRO               44
Pro               38
Pro               35
EnterPrise        13
ENTERPRISE        11
 enterprise        7
Enterprise         7
premium            1
vip                1
enterprise+        1
Name: count, dtype: int64

 post_category:
post_category
tech           118

## Chequeos logicos de fechas

In [17]:
#parsear fechas temporalmente para medir
#si da error se carga como nulo
df['created_at_tmp'] = pd.to_datetime(df['created_at'], errors='coerce')
df['post_created_at_tmp'] = pd.to_datetime(df['post_created_at'], errors='coerce')

#no parseables
no_parse_signup = df['created_at_tmp'].isnull().sum()
no_parse_post = df['post_created_at_tmp'].isnull().sum()
print(f"Fechas no parseables - created_at: {no_parse_signup}")
print(f"Fechas no parseables - post_created_at: {no_parse_post}")


#posts antes del signup (error logico)
post_antes = (df['post_created_at_tmp'] < df['created_at_tmp']).sum()
print(f"\n Posts antes del signup: {post_antes}")

#recalcular days_since_signup
df['days_since_signup_calc'] = (
    df['post_created_at_tmp'] - df['created_at_tmp']
).dt.days

#comparar con el original
diferencias = (df['days_since_signup'] != df['days_since_signup_calc']).sum()
print(f"Mismatches vs columna original: {diferencias}")

#ver algunos casos donde difieren
print("\n Ejemplos de mismatch:")
mask = df['days_since_signup'] != df['days_since_signup_calc']
display(df[mask][['user_id',
                  'created_at',
                  'post_created_at',
                  'days_since_signup',
                  'days_since_signup_calc']].head(10))


Fechas no parseables - created_at: 1
Fechas no parseables - post_created_at: 1

 Posts antes del signup: 100
Mismatches vs columna original: 4479

 Ejemplos de mismatch:


,user_id,created_at,post_created_at,days_since_signup,days_since_signup_calc
5,U00488,2025-03-19 16:56:05,2025-03-28 12:10:25,9,8.0
7,U01863,2024-12-12 21:40:02,2025-02-02 21:09:32,52,51.0
9,U00880,2024-12-12 23:03:33,2024-12-19 17:52:17,7,6.0
17,U01956,2025-02-19 10:47:45,2025-04-28 01:34:08,68,67.0
18,U01179,2024-08-28 23:29:24,2024-09-14 18:31:09,17,16.0
19,U01871,2025-07-15 19:58:22,2025-08-08 05:42:11,24,23.0
20,U00707,2024-10-07 17:32:12,2024-11-22 14:18:25,46,45.0
21,U00753,2024-02-10 23:28:54,2024-03-03 23:22:03,22,21.0
22,U01754,2025-01-17 07:05:21,2024-12-18 16:05:21,0,-30.0
26,U00485,2025-09-05 15:23:47,2025-09-24 01:27:51,19,18.0
